<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase8-multi-agent-governance/01_multi_agent_orchestration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 8: Multi-Agent Orchestration
**Goal**: Build a multi-agent compliance audit system with
      a governance controller monitoring specialized
      agents working together.
**Regulatory mapping**: EU AI Act Article 14 human oversight,
                    NIST AI RMF Govern and Manage functions.
**Date**: June 2026.
**Status**: In Progress

In [5]:
import time
import json
import re
import pandas as pd
from datetime import datetime
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def ask_llm(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model="gemini-flash-latest",
                contents=prompt
            )
            return response.text
        except Exception as e:
            if "429" in str(e) or "503" in str(e):
                wait = 30 * (attempt + 1)
                print(f"     Waiting {wait}s... (attempt {attempt+1}/{retries})")
                time.sleep(wait)
            else:
                raise e
    return "Error: max retries exceeded"

# - THE SYSTEM UNDER AUDIT -
# This represents an AI system being audited for compliance.
# In a real engagement this would be a client's AI system.
# Here I will define a system profile with known characteristics.

SYSTEM_UNDER_AUDIT = {
    "system_name":      "TalentMatch AI",
    "purpose":          "Automated CV screening and candidate ranking",
    "sector":           "employment",
    "deployment":       "EU and UK markets",
    "training_data":    "Historical hiring decisions 2015-2023",
    "known_metrics": {
        "gender_disparate_impact":   0.43,
        "age_disparate_impact":      0.20,
        "ethnicity_disparate_impact":0.80,
    },
    "human_oversight":  "None currently implemented",
    "documentation":    "Partial. No FRIA completed.",
}

# - SPECIALIZED AUDIT AGENTS -
# Each agent audits one specific compliance dimension

def bias_audit_agent(system):
  """Agent 1: Audits demographic bias under Article 10."""
  metrics = system["known_metrics"]
  findings = [] # Corrected typo: findinds -> findings
  critical = False

  for dimension, ratio in metrics.items():
    if ratio < 0.80:
      severity = "CRITICAL" if ratio < 0.50 else "HIGH"
      if severity == "CRITICAL":
        critical = True
      findings.append({
          "dimension": dimension,
          "ratio":     ratio,
          "severity":  severity,
          "compliant": False
      })
    else: # Corrected indentation of else statement
      findings.append({
          "dimension": dimension,
          "ratio":     ratio,
          "severity":  "PASS",
          "compliant": True
      })

  return {
      "agent":            "Bias Audit Agent",
      "article":          "EU AI Act Article 10",
      "critical_breach":   critical,
      "findings":          findings,
      "verdict":           "FAIL" if not all (
                           f["compliant"] for f in findings) else "PASS"
  }

def oversight_audit_agent(system):
  """Agent 2: Audits human oversight under Article 14."""
  oversight = system["human_oversight"].lower()
  has_oversight = "none" not in oversight and oversight != ""

  return {
      "agent":            "Oversight Audit Agent",
      "article":          "EU AI Act Article 14",
      "critical_breach":   not has_oversight,
      "findings":          system["human_oversight"],
      "verdict":           "PASS" if has_oversight else "FAIL"

  }

def documentation_audit_agent(system):
  """Agent 3: Audits documentation under Article 11 and 18."""
  docs = system["documentation"].lower()
  has_fria = "fria" in docs and "no fria" not in docs
  complete = "partial" not in docs and "no " not in docs

  return {
      "agent":            "Documentation Audit Agent",
      "article":          "EU AI Act Article 11 and 18",
      "finding":          system["documentation"],
      "critical_breach":  False,
      "verdict":          "PASS" if complete else "FAIL"
  }

# AGENT REGISTRY -
AUDIT_AGENTS = {
    "bias":            bias_audit_agent,
    "oversight":        oversight_audit_agent,
    "documentation":    documentation_audit_agent
}

print("====== MULTI-AGENT AUDIT SYSTEM READY ======")
print(f"Sytem under audit: {SYSTEM_UNDER_AUDIT['system_name']}")
print(f"Purpose:           {SYSTEM_UNDER_AUDIT['purpose']}")
print(f"Sector:            {SYSTEM_UNDER_AUDIT['sector']}")
print(f"\nAudit agents registered:   {len (AUDIT_AGENTS)}")
for name in AUDIT_AGENTS:
  print(f"   - {name} audit agent")
print("\nMulti_agent system ready \u2705")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== MULTI-AGENT AUDIT SYSTEM READY ======
Sytem under audit: TalentMatch AI
Purpose:           Automated CV screening and candidate ranking
Sector:            employment

Audit agents registered:   3
   - bias audit agent
   - oversight audit agent
   - documentation audit agent

Multi_agent system ready ✅


In [6]:
# - GOVERNANCE CONTROLLER -
# Orchestrates all audit agents and monitors for critical breaches.
# This is the layer that will trigger the kill-switch later.

def governance_controller(system, agents):
  """
  Runs all audit agents in sequence and monitors
  for critical breaches across the entire audit.
  """
  print("=" * 60)
  print(f"GOVERNANCE CONTROLLER: Auditing {system['system_name']}")
  print("=" * 60)

  audit_results = []
  critical_breaches = []

  for agent_name, agent_function in agents.items():
    print(f"\nRunning {agent_name} audit agent...")

    result = agent_function(system)
    audit_results.append(result)

    print(f" Agent: {result['agent']}")
    print(f" Article: {result['article']}")
    print(f" Verdict: {result['verdict']}")

    if result.get("critical_breach"):
      print(f"  *** CRITICAL BREACH DETECTED ***")
      critical_breaches.append({
          "agent": result["agent"],
          "article": result["article"],
      })

  # Overall assessment
  print("\n" + "=" * 60)
  print("GOVERNANCE CONTROLLER: Audit Summary")
  print("=" * 60)

  total_agents    = len(audit_results)
  passed          = sum(1 for r in audit_results
                        if r["verdict"] == "PASS")
  failed          = total_agents - passed
  has_critical    = len(critical_breaches) > 0

  print(f" Agents run:         {total_agents}")
  print(f" Passed:             {passed}")
  print(f" Failed:             {failed}")
  print(f" Critical breaches:  {len(critical_breaches)}")


  if has_critical:
    print(f"\n*** AUDIT VERDICT: CRITICAL NON-COMPLIANCE ***")
    print("Critical breaches detected in:")
    for breach in critical_breaches:
      print(f"  - {breach['agent']} ({breach['article']})")
    overall_verdict = "CRITICAL NON-COMPLIANCE"
  elif failed > 0:
    print(f"\n*** AUDIT VERDICT: NON-COMPLIANT ***")
    overall_verdict = "NON-COMPLIANT"
  else:
    print(f"\n*** AUDIT VERDICT: COMPLIANT ***")
    overall_verdict = "COMPLIANT"

  return {
      "system":             system["system_name"],
      "audit_results":      audit_results,
      "overall_verdict":    overall_verdict,
      "critical_breaches":  critical_breaches,
      "passed":             passed,
      "failed":             failed,
      "total_agents":       total_agents,
      "timestamp":          datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
  }

# - RUN THE MULTI-AGENT AUDIT -
audit = governance_controller(SYSTEM_UNDER_AUDIT, AUDIT_AGENTS)

print("\n" + "=" * 60)
print("Multi-agent orchestration complete ✅")
print(f"Overall verdict: {audit['overall_verdict']}")
print(f"Timestamp: {audit['timestamp']}")

GOVERNANCE CONTROLLER: Auditing TalentMatch AI

Running bias audit agent...
 Agent: Bias Audit Agent
 Article: EU AI Act Article 10
 Verdict: FAIL
  *** CRITICAL BREACH DETECTED ***

Running oversight audit agent...
 Agent: Oversight Audit Agent
 Article: EU AI Act Article 14
 Verdict: FAIL
  *** CRITICAL BREACH DETECTED ***

Running documentation audit agent...
 Agent: Documentation Audit Agent
 Article: EU AI Act Article 11 and 18
 Verdict: FAIL

GOVERNANCE CONTROLLER: Audit Summary
 Agents run:         3
 Passed:             0
 Failed:             3
 Critical breaches:  2

*** AUDIT VERDICT: CRITICAL NON-COMPLIANCE ***
Critical breaches detected in:
  - Bias Audit Agent (EU AI Act Article 10)
  - Oversight Audit Agent (EU AI Act Article 14)

Multi-agent orchestration complete ✅
Overall verdict: CRITICAL NON-COMPLIANCE
Timestamp: 2026-06-13 21:44:36
